# MT01 — Introduction to robosuite

### Lab Description

**robosuite** is a simulation framework for robot manipulation built on the MuJoCo physics engine. It bundles together everything you need for robot learning: **tasks** (e.g. lift a cube, open a door), **robots** (Panda, Sawyer, ...), **controllers** (joint torque, operational-space end-effector control), and **sensors** (proprioception, cameras). It is the PyTorch-side foundation of this course.

In this first lab you create a robosuite task, inspect what the robot observes and how it is commanded, and watch a random policy flail around — establishing the env / observation / action loop that everything else builds on. Everything runs headless on the AMD GPU via EGL.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + PyTorch environment (the `auplc-mujoco-torch` course image: torch + mujoco + robosuite + gymnasium) that this notebook is built for.

## Goals
- Create a robosuite manipulation task with a Panda arm
- Understand robosuite's dict observations and 7-D action
- Render a robosuite scene reliably on this ROCm/EGL stack
- Roll out a random policy and save an inline video

### Imports and a reliable renderer

We set the EGL backend before importing MuJoCo, then define two helpers. Note the renderer comment: robosuite's own camera-observation path is occasionally unstable here, so we render with `mujoco.Renderer` and show only the visual geom group. You will reuse `make_renderer` / `grab_frame` in every MT notebook.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import numpy as np
import mujoco
import imageio
import robosuite as suite
from robosuite import load_composite_controller_config
from IPython.display import Video

import logging
logging.getLogger("robosuite_logs").setLevel(logging.ERROR)  # hide repetitive controller-config warnings

os.makedirs("output/videos", exist_ok=True)
print("robosuite", suite.__version__, "| mujoco", mujoco.__version__)

# robosuite's built-in camera-observation renderer can emit intermittently
# corrupted frames on this ROCm/EGL stack. We instead drive mujoco.Renderer
# directly (the reliable path) and show only the visual geom group (group 1),
# which avoids the green/blue collision shapes and looks correct.
def make_renderer(env, height=256, width=256):
    R = mujoco.Renderer(env.sim.model._model, height=height, width=width)
    opt = mujoco.MjvOption()
    opt.geomgroup[:] = 0
    opt.geomgroup[1] = 1
    return R, opt

def grab_frame(env, R, opt, camera="frontview"):
    mujoco.mj_forward(env.sim.model._model, env.sim.data._data)
    R.update_scene(env.sim.data._data, camera=camera, scene_option=opt)
    return R.render()

### Create a task

`suite.make` assembles a task. We pick the **Lift** task (lift a cube) with a **Panda** arm and the **BASIC** controller (operational-space end-effector control). We keep observations low-dimensional (`use_camera_obs=False`) and render ourselves, which is both stable and fast.

In [ ]:
controller = load_composite_controller_config(controller="BASIC", robot="Panda")
env = suite.make(
    env_name="Lift",               # task: lift a cube off the table
    robots="Panda",                # 7-DoF Franka Panda arm
    controller_configs=controller,
    has_renderer=False,            # no on-screen window (headless)
    has_offscreen_renderer=False,  # we render ourselves with mujoco.Renderer
    use_camera_obs=False,          # observations are low-dim state, not pixels
    control_freq=20,
    horizon=200,
)
obs = env.reset()
print("number of observation keys:", len(obs))
print("action dimension          :", env.action_dim)

### Observations and actions

robosuite returns a **dict** of named observations. For the Lift task the useful ones are the robot's joint angles, end-effector position, gripper opening, and the cube position. The **action** is a 7-vector for the Panda under the BASIC controller:
- `action[0:3]` — end-effector position delta (dx, dy, dz)
- `action[3:6]` — end-effector orientation delta (axis-angle)
- `action[6]` — gripper command (−1 open, +1 close)

In [ ]:
for k in ["robot0_joint_pos", "robot0_eef_pos", "robot0_gripper_qpos", "cube_pos"]:
    print(f"{k:22s} shape {np.asarray(obs[k]).shape}  value {np.round(obs[k], 3)}")

low, high = env.action_spec
print("\naction low :", np.round(low, 2))
print("action high:", np.round(high, 2))

### Roll out a random policy

With no controller logic, random actions just jitter the arm. We step the env for 150 steps with uniform-random actions, grab a frame each step, and save an mp4. The total reward stays near zero — random motor babbling rarely lifts the cube. (Later notebooks add control and learning.)

In [ ]:
R, opt = make_renderer(env, 256, 256)
obs = env.reset()
frames = [grab_frame(env, R, opt, camera="frontview")]
total_reward = 0.0
for _ in range(150):
    action = np.random.uniform(-1, 1, env.action_dim)
    obs, reward, done, info = env.step(action)
    total_reward += reward
    frames.append(grab_frame(env, R, opt, camera="frontview"))
R.close()
print(f"random policy total reward: {total_reward:.2f}")
imageio.mimsave("output/videos/mt01_random_rollout.mp4", frames, fps=20)
print("saved", len(frames), "frames")

### Watch the random rollout

In [ ]:
Video(url="output/videos/mt01_random_rollout.mp4")

## Conclusions

You built a robosuite manipulation task, saw its dict observations and 7-D end-effector action, set up a stable renderer for this ROCm/EGL stack, and rolled out a random policy. Random actions get nowhere — in MT02 we use the operational-space controller to *script* a successful pick.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT